In [1]:
# Upload kaggle.json manually when prompted
from google.colab import files
files.upload()

# Move kaggle.json to the correct folder
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API authenticated!")

Saving kaggle.json to kaggle.json
Kaggle API authenticated!


In [2]:
# Create datasets folder
!mkdir -p /content/datasets

# Download the VITON-HD dataset (replace dataset name if needed)
!kaggle datasets download -d marquis03/high-resolution-viton-zalando-dataset -p /content/datasets

Dataset URL: https://www.kaggle.com/datasets/marquis03/high-resolution-viton-zalando-dataset
License(s): CC-BY-NC-SA-4.0


In [3]:
# Unzip it
!unzip -q /content/datasets/high-resolution-viton-zalando-dataset.zip -d /content/datasets/viton_hd

print("VITON-HD dataset downloaded and extracted!")

VITON-HD dataset downloaded and extracted!


In [4]:
!pip install ftfy regex
!pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-2uyywrb3
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-2uyywrb3
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import os
import json
import numpy as np
import torch
import clip
from torch import nn, optim
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from collections import defaultdict

In [6]:
# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_PATH = "/content/datasets/viton_hd/test"
TEST_PAIRS = "/content/datasets/viton_hd/test_pairs.txt"
SAVE_DIR = "artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

In [7]:
# --- Helper Functions ---
def load_image(path):
    return Image.open(path).convert('RGB')

def extract_clip_features(model, preprocess, image):
    image_input = preprocess(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        features = model.encode_image(image_input)
    return features.cpu().numpy().flatten()

def cosine_similarity(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return np.dot(a, b)

def retrieve_topk(query_feat, gallery_feats, gallery_paths, topk=5):
    sims = [cosine_similarity(query_feat, feat) for feat in gallery_feats]
    topk_idx = np.argsort(sims)[::-1][:topk]
    return [gallery_paths[i] for i in topk_idx]

def compute_accuracy(lst, total_queries):
    return (np.sum(lst) / total_queries) * 100 if total_queries > 0 else 0

def visualize_tsne(features, labels, save_path):
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    reduced = tsne.fit_transform(features)
    plt.figure(figsize=(10, 7))
    for label in np.unique(labels):
        idx = np.where(labels == label)
        plt.scatter(reduced[idx, 0], reduced[idx, 1], label=label)
    plt.legend()
    plt.savefig(save_path)
    plt.close()

In [8]:
# --- Retrieval Head ---
class RetrievalHead(nn.Module):
    def __init__(self, input_dim, output_dim=512):
        super(RetrievalHead, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.ReLU(),
            nn.Linear(output_dim, output_dim)
        )

    def forward(self, x):
        return self.fc(x)

In [9]:
# --- Triplet Dataset ---
class TripletDataset(Dataset):
    def __init__(self, pairs, cloth_paths, cloth_feats, clip_model, clip_preprocess):
        self.pairs = pairs
        self.cloth_paths = cloth_paths
        self.cloth_feats = cloth_feats
        self.clip_model = clip_model
        self.clip_preprocess = clip_preprocess

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        anchor_img_name, positive_cloth_name = self.pairs[idx]
        anchor_img_path = os.path.join(DATASET_PATH, "image", anchor_img_name)
        anchor_img = load_image(anchor_img_path)
        anchor_feat = extract_clip_features(self.clip_model, self.clip_preprocess, anchor_img)

        positive_idx = [i for i, p in enumerate(self.cloth_paths) if os.path.basename(p) == positive_cloth_name][0]
        positive_feat = self.cloth_feats[positive_idx]

        neg_idx = np.random.choice(len(self.cloth_paths))
        negative_feat = self.cloth_feats[neg_idx]

        return torch.tensor(anchor_feat).float(), torch.tensor(positive_feat).float(), torch.tensor(negative_feat).float()

In [10]:
# --- Training Function ---
def train_retrieval_head(train_loader, retrieval_head, epochs=10, lr=1e-4):
    optimizer = optim.Adam(retrieval_head.parameters(), lr=lr)
    criterion = nn.TripletMarginLoss(margin=0.3)
    retrieval_head.train()

    for epoch in range(epochs):
        epoch_loss = 0
        for anchor, positive, negative in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            anchor, positive, negative = anchor.to(DEVICE), positive.to(DEVICE), negative.to(DEVICE)
            anchor_out = retrieval_head(anchor)
            positive_out = retrieval_head(positive)
            negative_out = retrieval_head(negative)

            loss = criterion(anchor_out, positive_out, negative_out)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_loader):.4f}")

In [11]:
# --- Loading CLIP Model ---
clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE)
for param in clip_model.parameters():
    param.requires_grad = False

100%|███████████████████████████████████████| 338M/338M [00:16<00:00, 21.3MiB/s]


In [12]:
# --- Preparing Dataset ---
with open(TEST_PAIRS, 'r') as f:
    lines = f.readlines()
pairs = [line.strip().split() for line in lines]

cloth_dir = os.path.join(DATASET_PATH, "cloth")
cloth_paths = sorted([os.path.join(cloth_dir, c) for c in os.listdir(cloth_dir) if c.endswith('.jpg')])
cloth_clip_feats = []
for p in tqdm(cloth_paths, desc="Extracting CLIP features for clothes"):
    cloth_clip_feats.append(extract_clip_features(clip_model, clip_preprocess, load_image(p)))

Extracting CLIP features for clothes: 100%|██████████| 2032/2032 [00:47<00:00, 43.03it/s]


In [13]:
# --- Fine-tuning Retrieval Head ---
train_dataset = TripletDataset(pairs, cloth_paths, cloth_clip_feats, clip_model, clip_preprocess)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

retrieval_head = RetrievalHead(input_dim=512).to(DEVICE)
train_retrieval_head(train_loader, retrieval_head, epochs=10)

torch.save(retrieval_head.state_dict(), os.path.join(SAVE_DIR, "retrieval_head.pth"))
print("Retrieval head fine-tuned and saved.")

Epoch 1/10: 100%|██████████| 64/64 [00:51<00:00,  1.25it/s]


Epoch 1 Loss: 0.3131


Epoch 2/10: 100%|██████████| 64/64 [00:48<00:00,  1.33it/s]


Epoch 2 Loss: 0.2934


Epoch 3/10: 100%|██████████| 64/64 [00:48<00:00,  1.32it/s]


...

Epoch 8 Loss: 0.2314


Epoch 9/10: 100%|██████████| 64/64 [00:47<00:00,  1.33it/s]


Epoch 9 Loss: 0.2232


Epoch 10/10: 100%|██████████| 64/64 [00:48<00:00,  1.33it/s]

Epoch 10 Loss: 0.2072
Retrieval head fine-tuned and saved.


In [19]:
# 📊 Retrieval Metrics Functions
from sklearn.metrics import average_precision_score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Retrieval Metrics
def precision_at_k(actual, predicted, k):
    actual_set = set(actual)
    predicted = predicted[:k]
    return len(set(predicted) & actual_set) / len(predicted)

def recall_at_k(actual, predicted, k):
    actual_set = set(actual)
    predicted = predicted[:k]
    return len(set(predicted) & actual_set) / len(actual_set)

def mean_average_precision(actual_list, predicted_list, k=5):
    average_precisions = []
    for actual, predicted in zip(actual_list, predicted_list):
        ap = 0.0
        correct = 0
        for i in range(min(k, len(predicted))):
            if predicted[i] in actual:
                correct += 1
                ap += correct / (i + 1)
        if len(actual) > 0:
            average_precisions.append(ap / min(len(actual), k))
    return sum(average_precisions) / len(average_precisions)

def compute_retrieval_metrics(actual_labels, predicted_labels, k_list=[1,3,5]):
    for k in k_list:
        precisions = [precision_at_k(a, p, k) for a, p in zip(actual_labels, predicted_labels)]
        recalls = [recall_at_k(a, p, k) for a, p in zip(actual_labels, predicted_labels)]

        print(f"Precision@{k}: {sum(precisions)/len(precisions):.4f}")
        print(f"Recall@{k}: {sum(recalls)/len(recalls):.4f}")

    map_score = mean_average_precision(actual_labels, predicted_labels, k=max(k_list))
    print(f"Mean Average Precision (mAP@{max(k_list)}): {map_score:.4f}")

# t-SNE Evaluation Metrics
def evaluate_tsne_clusters(tsne_embeddings, n_clusters=5):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(tsne_embeddings)

    silhouette = silhouette_score(tsne_embeddings, cluster_labels)
    davies_bouldin = davies_bouldin_score(tsne_embeddings, cluster_labels)
    calinski_harabasz = calinski_harabasz_score(tsne_embeddings, cluster_labels)

    print(f"Silhouette Score: {silhouette:.4f}")
    print(f"Davies-Bouldin Index: {davies_bouldin:.4f} (lower is better)")
    print(f"Calinski-Harabasz Score: {calinski_harabasz:.4f} (higher is better)")

In [20]:
# --- Evaluating Fine-tuned Model ---
retrieval_head.eval()

cloth_retrieval_feats = []
for feat in cloth_clip_feats:
    with torch.no_grad():
        feat_tensor = torch.tensor(feat).float().unsqueeze(0).to(DEVICE)
        retrieval_feat = retrieval_head(feat_tensor)
        cloth_retrieval_feats.append(retrieval_feat.squeeze(0).cpu().numpy())

# Group ground-truth cloths per person
person_to_cloths = defaultdict(list)
for person_img_name, cloth_img_name in pairs:
    person_to_cloths[person_img_name].append(cloth_img_name)

clip_top1, clip_top3, clip_top5 = [], [], []

all_actual_labels = []
all_predicted_labels = []

for person_img_name, gt_cloth_names in tqdm(person_to_cloths.items(), desc="Evaluating test persons"):
    person_path = os.path.join(DATASET_PATH, "image", person_img_name)
    person_img = load_image(person_path)

    person_clip_feat_raw = extract_clip_features(clip_model, clip_preprocess, person_img)
    with torch.no_grad():
        person_clip_feat_tensor = torch.tensor(person_clip_feat_raw).float().unsqueeze(0).to(DEVICE)
        person_feat = retrieval_head(person_clip_feat_tensor).squeeze(0).cpu().numpy()

    retrieved = retrieve_topk(person_feat, cloth_retrieval_feats, cloth_paths, topk=10)

    retrieved_names = []
    for p in retrieved:
        name = os.path.basename(p)
        if name not in retrieved_names:
            retrieved_names.append(name)
        if len(retrieved_names) == 5:  # Top-5 predictions only
            break

    # Existing evaluation
    clip_top1.append(any(gt == retrieved_names[0] for gt in gt_cloth_names))
    clip_top3.append(any(gt in retrieved_names[:3] for gt in gt_cloth_names))
    clip_top5.append(any(gt in retrieved_names[:5] for gt in gt_cloth_names))

    # Collect for retrieval metrics
    all_actual_labels.append(gt_cloth_names)   # Ground-truth cloth(s)
    all_predicted_labels.append(retrieved_names)  # Top-5 retrieved cloth names


Evaluating test persons: 100%|██████████| 2032/2032 [01:28<00:00, 22.88it/s]


In [21]:
print("\nComputing Retrieval Metrics:")
compute_retrieval_metrics(all_actual_labels, all_predicted_labels)


Computing Retrieval Metrics:
Precision@1: 0.0025
Recall@1: 0.0025
Precision@3: 0.0016
Recall@3: 0.0049
Precision@5: 0.0023
Recall@5: 0.0113
Mean Average Precision (mAP@5): 0.0049


In [22]:
# --- Metrics Calculation ---
num_queries = len(person_to_cloths)
clip_results = {
    "Top-1": compute_accuracy(clip_top1, num_queries),
    "Top-3": compute_accuracy(clip_top3, num_queries),
    "Top-5": compute_accuracy(clip_top5, num_queries),
}

with open(os.path.join(SAVE_DIR, "fine_tuned_clip_results.json"), 'w') as f:
    json.dump(clip_results, f, indent=4)

print("Fine-tuned retrieval evaluation completed and corrected results saved.")

# --- t-SNE Visualization ---
all_feats = np.vstack(cloth_retrieval_feats)
all_labels = np.array(["cloth"] * len(cloth_retrieval_feats))
visualize_tsne(all_feats, all_labels, os.path.join(SAVE_DIR, "tsne_retrieval_head.png"))
print("t-SNE visualization saved.")

Fine-tuned retrieval evaluation completed and corrected results saved.
t-SNE visualization saved.


In [23]:
print("\nComputing t-SNE Clustering Metrics:")
evaluate_tsne_clusters(np.vstack(cloth_retrieval_feats))


Computing t-SNE Clustering Metrics:
Silhouette Score: 0.0558
Davies-Bouldin Index: 2.9187 (lower is better)
Calinski-Harabasz Score: 105.5781 (higher is better)
